In [ ]:
import pandas as pd
import pyodbc
import json
from datetime import date, datetime
from nbformat import read

# Charger configuration JSON
with open("staging_config.json", encoding="utf-8") as f:
    config_list = json.load(f)

# Connexion SQL Server (adapter)
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Staging_Test;"
    "UID=moeness;"
    "PWD=azerty"
)


In [ ]:
# Dictionnaire des tables référentielles
referentiels = {
    "Postes": pd.read_sql("SELECT * FROM Postes", conn),
    "Categories": pd.read_sql("SELECT * FROM Categories", conn),
    "Pieds": pd.read_sql("SELECT * FROM Pieds", conn),
    "Competitions": pd.read_sql("SELECT * FROM Competitions", conn),
    "Meteo": pd.read_sql("SELECT * FROM Meteo", conn),
    "Etats_Terrain": pd.read_sql("SELECT * FROM Etats_Terrain", conn),
    "Resultats": pd.read_sql("SELECT * FROM Resultats", conn)
}


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Postes": pd.read_sql("SELECT * FROM Postes", conn),
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Categories": pd.read_sql("SELECT * FROM Categories", conn),
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_3468\3942123766.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  "Pieds": pd.read_sql("SELECT * FROM Pieds", conn),
C:\Users\MOEµNESS\A

In [ ]:
def load_and_historize_with_log(conn, config, referentiels, log_path="staging_log.txt"):
    table = config["table"]
    csv_path = config["csv"]
    id_col = config["id_col"]
    compare_cols = config["compare_cols"].copy()
    dtypes = config.get("dtypes")
    today = date.today()
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def log(msg):
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"[{now_str}] 📥 Table: {table} → {msg}\n")

    if not dtypes:
        msg = "❌ Erreur : 'dtypes' manquant dans le JSON."
        print(msg)
        log(msg)
        return

    # 🧱 Créer la table si elle n'existe pas
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = ?", (table,))
    if not cursor.fetchone()[0]:
        try:
            column_defs = [f"{id_col} {dtypes.get(id_col, 'INT')}"]
            for col in compare_cols:
                if col != id_col:
                    column_defs.append(f"{col} {dtypes.get(col, 'VARCHAR(255)')}")
            column_defs += [
                "Date_Debut DATE",
                "Date_Fin DATE NULL",
                "Last_Modified DATETIME NULL",
                "Actif BIT",
                "Type_Changement VARCHAR(20)"
            ]
            column_sql = ",\n  ".join(column_defs)
            create_stmt = f"""
                CREATE TABLE {table} (
                  {column_sql}
                    )
                """
            # add ,CONSTRAINT PK_{table}_{id_col} PRIMARY KEY ({id_col})  to create_stmt if we want a primary key
            cursor.execute(create_stmt)
            conn.commit()
            log("🆕 Table créée avec succès.")
        except Exception as e:
            msg = f"❌ Erreur création table : {e}"
            print(msg)
            log(msg)
            return

    # 📤 Charger le CSV
    try:
        df = pd.read_csv(csv_path, parse_dates=["Date_Examen"] if "Date_Examen" in compare_cols else [])
        df = df.astype("object")
    except Exception as e:
        msg = f"❌ Erreur lecture CSV : {e}"
        print(msg)
        log(msg)
        return

    # 🔁 Mapping référentiels
    #if table == "Joueurs":
     #   if "Position" in compare_cols:
     #       df = df.merge(referentiels["Postes"], how="left", left_on="Position", right_on="Nom_Poste")
     #       df["ID_Poste"] = df["ID_Poste"].astype(int)
      #      compare_cols.remove("Position")
       #     compare_cols.append("ID_Poste")
        #if "Catégorie" in compare_cols:
         #   df = df.merge(referentiels["Categories"], how="left", left_on="Catégorie", right_on="Nom_Categorie")
          #  df["ID_Categorie"] = df["ID_Categorie"].astype(int)
           # compare_cols.remove("Catégorie")
            #compare_cols.append("ID_Categorie")
        #if "Pied_Dominant" in compare_cols:
         #   df = df.merge(referentiels["Pieds"], how="left", left_on="Pied_Dominant", right_on="Cote_Pied")
          #  df["ID_Pied"] = df["ID_Pied"].astype(int)
           # compare_cols.remove("Pied_Dominant")
            #compare_cols.append("ID_Pied") **/

    if table == "Matchs":
        if "Compétition" in compare_cols:
            df = df.merge(referentiels["Competitions"], how="left", left_on="Compétition", right_on="Nom_Competition")
            df["ID_Competition"] = df["ID_Competition"].astype(int)
            compare_cols.remove("Compétition")
            compare_cols.append("ID_Competition")
        if "Résultat" in compare_cols:
            df = df.merge(referentiels["Resultats"], how="left", left_on="Résultat", right_on="Nom_Resultat")
            df["ID_Resultat"] = df["ID_Resultat"].astype(int)
            compare_cols.remove("Résultat")
            compare_cols.append("ID_Resultat")
        if "Catégorie" in compare_cols:
            df = df.merge(referentiels["Categories"], how="left", left_on="Catégorie", right_on="Nom_Categorie")
            df["ID_Categorie"] = df["ID_Categorie"].astype(int)
            compare_cols.remove("Catégorie")
            compare_cols.append("ID_Categorie")

    elif table == "Donnees_Contextuelles":
        if "Météo" in compare_cols:
            df = df.merge(referentiels["Meteo"], how="left", left_on="Météo", right_on="Type_Meteo")
            df["ID_Meteo"] = df["ID_Meteo"].astype(int)
            compare_cols.remove("Météo")
            compare_cols.append("ID_Meteo")
        if "Etat_Terrain" in compare_cols:
            df = df.merge(referentiels["Etats_Terrain"], how="left", left_on="Etat_Terrain", right_on="Etat")
            df["ID_Etat"] = df["ID_Etat"].astype(int)
            compare_cols.remove("Etat_Terrain")
            compare_cols.append("ID_Etat")
        if "Compétition" in compare_cols:
            df = df.merge(referentiels["Competitions"], how="left", left_on="Compétition", right_on="Nom_Competition")
            df["ID_Competition"] = df["ID_Competition"].astype(int)
            compare_cols.remove("Compétition")
            compare_cols.append("ID_Competition")

    try:
        df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)
    except Exception as e:
        msg = f"❌ Erreur lecture table SQL : {e}"
        print(msg)
        log(msg)
        return

    inserted, updated = 0, 0
    for _, row in df.iterrows():
        old = df_db[df_db[id_col] == row[id_col]]
        if old.empty:
            inserted += 1
            cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
            placeholders = ", ".join(["?"] * (len(compare_cols) + 3 + 1))
            values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'INSERT']
            cursor.execute(f"INSERT INTO {table} ({cols_sql}) VALUES ({placeholders})", *values)
        else:
            old_row = old.iloc[0]
            if any(str(row[col]) != str(old_row[col]) for col in compare_cols):
                updated += 1
                cursor.execute(f"""
                    UPDATE {table}
                    SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                    WHERE {id_col} = ? AND Actif = 1
                """, today, datetime.now(), row[id_col])
                cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
                placeholders = ", ".join(["?"] * (len(compare_cols) + 3 + 1))
                values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'UPDATE']
                cursor.execute(f"INSERT INTO {table} ({cols_sql}) VALUES ({placeholders})", *values)

    conn.commit()
    cursor.close()
    msg = f"✔️ Insertion: {inserted}, Mise à jour: {updated}"
    print(msg)
    log(msg)